In [ ]:

courses = []    
next_id = [1]  


UNWEIGHTED = {"A": 4, "B": 3, "C": 2, "D": 1, "F": 0}
WEIGHTED = {
    "Regular": {"A": 4, "B": 3, "C": 2, "D": 1, "F": 0},
    "Honors":  {"A": 5, "B": 4, "C": 3, "D": 2, "F": 1},
    "AP":      {"A": 5, "B": 4, "C": 3, "D": 2, "F": 1},
}


def calculate_gpa(course_list):
    if len(course_list) == 0:
        return None, None
    uw_total = 0  
    w_total  = 0  

    for course in course_list:
        uw_total += UNWEIGHTED[course["grade"]]
        w_total  += WEIGHTED[course["type"]][course["grade"]]

    unweighted_gpa = round(uw_total / len(course_list), 2)
    weighted_gpa   = round(w_total  / len(course_list), 2)

    return unweighted_gpa, weighted_gpa

def calculate_and_display():
    uw, w = calculate_gpa(courses)  

    uw_el   = document.getElementById("uw-gpa")
    w_el    = document.getElementById("w-gpa")
    sum_el  = document.getElementById("summary")
    sum_txt = document.getElementById("summary-text")

    if uw is None:
        uw_el.textContent    = "—"
        w_el.textContent     = "—"
        sum_el.style.display = "none"
    else:
        uw_el.textContent = f"{uw:.2f}"
        w_el.textContent  = f"{w:.2f}"

        ap_count     = sum(1 for c in courses if c["type"] == "AP")
        honors_count = sum(1 for c in courses if c["type"] == "Honors")
        boost        = round(w - uw, 2)
        count        = len(courses)

        parts = [f"{count} course{'s' if count != 1 else ''}"]
        if ap_count:     parts.append(f"{ap_count} AP")
        if honors_count: parts.append(f"{honors_count} Honors")
        parts.append(f"weighting boost +{boost:.2f}")

        sum_txt.textContent  = "  ·  ".join(parts)
        sum_el.style.display = "block"


def render_courses():
    list_el = document.getElementById("courses-list")
    list_el.innerHTML = ""

    if len(courses) == 0:
        empty = document.createElement("div")
        empty.className = "empty-state"
        empty.id = "empty-msg"
        empty.textContent = "No courses yet — add one below"
        list_el.appendChild(empty)
        return

    for course in courses:
        cid = course["id"]

        row = document.createElement("div")
        row.className = "course-row"
        row.id = f"course-{cid}"

        name_input = document.createElement("input")
        name_input.type = "text"
        name_input.placeholder = "e.g. English"
        name_input.value = course["name"]

        def make_name_handler(course_id):
            def handler(ev):
                update_field(course_id, "name", ev.target.value)
            return handler
        name_input.addEventListener("input", create_proxy(make_name_handler(cid)))


        grade_sel = document.createElement("select")
        for g in ["A", "B", "C", "D", "F"]:
            opt = document.createElement("option")
            opt.value = g
            opt.textContent = g
            if course["grade"] == g:
                opt.selected = True
            grade_sel.appendChild(opt)

        def make_grade_handler(course_id):
            def handler(ev):
                update_field(course_id, "grade", ev.target.value)
            return handler
        grade_sel.addEventListener("change", create_proxy(make_grade_handler(cid)))

        type_sel = document.createElement("select")
        for t in ["Regular", "Honors", "AP"]:
            opt = document.createElement("option")
            opt.value = t
            opt.textContent = t
            if course["type"] == t:
                opt.selected = True
            type_sel.appendChild(opt)

        def make_type_handler(course_id):
            def handler(ev):
                update_field(course_id, "type", ev.target.value)
            return handler
        type_sel.addEventListener("change", create_proxy(make_type_handler(cid)))

        del_btn = document.createElement("button")
        del_btn.className = "del-btn"
        del_btn.textContent = "×"
        del_btn.title = "Remove"

        def make_del_handler(course_id):
            def handler(ev):
                remove_course(course_id)
            return handler
        del_btn.addEventListener("click", create_proxy(make_del_handler(cid)))

        row.appendChild(name_input)
        row.appendChild(grade_sel)
        row.appendChild(type_sel)
        row.appendChild(del_btn)
        list_el.appendChild(row)

def add_course(event=None):
    cid = next_id[0]
    next_id[0] += 1
    courses.append({"id": cid, "name": "", "grade": "A", "type": "Regular"})
    render_courses()
    calculate_and_display()

def remove_course(course_id):
    for i, c in enumerate(courses):
        if c["id"] == course_id:
            courses.pop(i)
            break
    render_courses()
    calculate_and_display()

def update_field(course_id, field, value):
    for c in courses:
        if c["id"] == course_id:
            c[field] = value  
            break
    calculate_and_display()

document.getElementById("add-btn").addEventListener("click", create_proxy(add_course))

add_course()
add_course()
add_course()